In [9]:
import os
import glob
from lxml import etree
import pandas as pd

PATH_XML_15 = "../data/raw/15-xml/compteRendu/"
PATH_XML_16 = "../data/raw/16-xml/compteRendu/"

ns = {"ns": "http://schemas.assemblee-nationale.fr/referentiel"}


def compter_italique_vides(dossier):
    total_vides, total_avec_contenu, fichiers_concernes = 0, 0, set()
    for fichier in glob.glob(os.path.join(dossier, "*.xml")):
        tree = etree.parse(fichier)
        for it in tree.getroot().iter(
            "{http://schemas.assemblee-nationale.fr/referentiel}italique"
        ):
            est_vide = not (it.text and it.text.strip()) and len(it) == 0
            if est_vide:
                total_vides += 1
                fichiers_concernes.add(os.path.basename(fichier))
            else:
                total_avec_contenu += 1
    return total_vides, total_avec_contenu, fichiers_concernes


vides_16, contenu_16, fichiers_16 = compter_italique_vides(PATH_XML_16)
vides_15, contenu_15, fichiers_15 = compter_italique_vides(PATH_XML_15)

print(
    f"16e législature : {vides_16} <italique/> vides, {contenu_16} avec contenu, {len(fichiers_16)} fichiers concernés"
)
print(
    f"15e législature : {vides_15} <italique/> vides, {contenu_15} avec contenu, {len(fichiers_15)} fichiers concernés"
)
print("== nb occurrences, pas nb paragraphe (voir après, éviter confusion)")

16e législature : 616 <italique/> vides, 109465 avec contenu, 335 fichiers concernés
15e législature : 74198 <italique/> vides, 316399 avec contenu, 1351 fichiers concernés
== nb occurrences, pas nb paragraphe (voir après, éviter confusion)


In [10]:
# 15eme

ns_uri = "http://schemas.assemblee-nationale.fr/referentiel"
ns = {"ns": ns_uri}


def texte_sans_fix(texte_elem):
    return "".join(texte_elem.itertext()).strip()


def texte_avec_fix(texte_elem_original):
    # travaille sur une copie pour ne pas polluer l'original utilisé par texte_sans_fix
    from copy import deepcopy

    texte_elem = deepcopy(texte_elem_original)
    for br in texte_elem.findall(".//ns:br", namespaces=ns):
        br.tail = " " + (br.tail or "")
    for it in texte_elem.findall(".//ns:italique", namespaces=ns):
        vide = not (it.text and it.text.strip()) and len(it) == 0
        if vide:
            it.tail = " " + (it.tail or "")
    return "".join(texte_elem.itertext()).strip()


nb_paragraphes_avec_italique_vide = 0
nb_paragraphes_reellement_modifies = 0
exemples_diff = []

for fichier in glob.glob(os.path.join(PATH_XML_15, "*.xml")):
    tree = etree.parse(fichier)
    for texte_elem in tree.getroot().iter(f"{{{ns_uri}}}texte"):
        a_italique_vide = any(
            not (it.text and it.text.strip()) and len(it) == 0
            for it in texte_elem.findall(".//ns:italique", namespaces=ns)
        )
        if a_italique_vide:
            nb_paragraphes_avec_italique_vide += 1
            avant = texte_sans_fix(texte_elem)
            apres = texte_avec_fix(texte_elem)
            if avant != apres:
                nb_paragraphes_reellement_modifies += 1
                if len(exemples_diff) < 5:
                    # exemples_diff.append((avant[:150], apres[:150]))
                    exemples_diff.append((avant, apres)) # affichage complet


print(
    f"Paragraphes contenant un <italique/> vide : {nb_paragraphes_avec_italique_vide}"
)
print(
    f"Paragraphes réellement modifiés par le fix : {nb_paragraphes_reellement_modifies}"
)
for avant, apres in exemples_diff:
    print(f"\nAVANT : {avant}\nAPRÈS : {apres}")

Paragraphes contenant un <italique/> vide : 43681
Paragraphes réellement modifiés par le fix : 42961

AVANT : J’ai hâte d’être au Sénat ! Dommage que les députés ne votent pas là-bas ! (Rires etapplaudissements sur divers bancs.)
APRÈS : J’ai hâte d’être au Sénat ! Dommage que les députés ne votent pas là-bas ! (Rires et applaudissements sur divers bancs.)

AVANT : Trois points.D’abord, je veux vous remercier, monsieur le ministre, car c’est la première fois que vous reconnaissez qu’un certain nombre d’offices ne sont pas d’accord avec ces regroupements. Pour ma part, je crois qu’ils sont encore plus nombreux que vous l’imaginez.Ensuite, votre liberté de visiter Lunéville la semaine prochaine avec notre excellent collègue Thibault Bazin dépend certes un peu de nous, mais, comme nous l’avons dit hier soir, ce n’est pas notre faute si nous subissons les conséquences de l’organisation actuelle du travail parlementaire. Après avoir eu deux mois, au début de l’année, au cours desquels nous 

In [11]:
# 16 eme
ns_uri = "http://schemas.assemblee-nationale.fr/referentiel"
ns = {"ns": ns_uri}


def texte_sans_fix(texte_elem):
    return "".join(texte_elem.itertext()).strip()


def texte_avec_fix(texte_elem_original):
    # travaille sur une copie pour ne pas polluer l'original utilisé par texte_sans_fix
    from copy import deepcopy

    texte_elem = deepcopy(texte_elem_original)
    for br in texte_elem.findall(".//ns:br", namespaces=ns):
        br.tail = " " + (br.tail or "")
    for it in texte_elem.findall(".//ns:italique", namespaces=ns):
        vide = not (it.text and it.text.strip()) and len(it) == 0
        if vide:
            it.tail = " " + (it.tail or "")
    return "".join(texte_elem.itertext()).strip()


nb_paragraphes_avec_italique_vide = 0
nb_paragraphes_reellement_modifies = 0
exemples_diff = []

for fichier in glob.glob(os.path.join(PATH_XML_16, "*.xml")):
    tree = etree.parse(fichier)
    for texte_elem in tree.getroot().iter(f"{{{ns_uri}}}texte"):
        a_italique_vide = any(
            not (it.text and it.text.strip()) and len(it) == 0
            for it in texte_elem.findall(".//ns:italique", namespaces=ns)
        )
        if a_italique_vide:
            nb_paragraphes_avec_italique_vide += 1
            avant = texte_sans_fix(texte_elem)
            apres = texte_avec_fix(texte_elem)
            if avant != apres:
                nb_paragraphes_reellement_modifies += 1
                if len(exemples_diff) < 5:
                    # exemples_diff.append((avant[:150], apres[:150]))
                    exemples_diff.append((avant, apres)) # affichage complet

print(
    f"Paragraphes contenant un <italique/> vide : {nb_paragraphes_avec_italique_vide}"
)
print(
    f"Paragraphes réellement modifiés par le fix : {nb_paragraphes_reellement_modifies}"
)
for avant, apres in exemples_diff:
    print(f"\nAVANT : {avant}\nAPRÈS : {apres}")

Paragraphes contenant un <italique/> vide : 587
Paragraphes réellement modifiés par le fix : 576

AVANT : Nous devons donc y remédier.Dans son discours de Belfort, le Président de la République a réaffirmé l’ambition de « faire […] de la France, le premier grand pays à sortir des énergies fossiles », grâce à la construction de six nouveaux réacteurs. Le nucléaire, énergie décarbonée, est un atout pour la France. Nous avons été nombreux à saluer l’engagement politique très fort d’Emmanuel Macron en faveur d’une filière modernisée, qui contribue à l’émergence et à l’image des savoir-faire français.Le nucléaire est la première source de production et de consommation d’électricité. Le parc français est le plus puissant au monde. Après celui des États-Unis, il est aussi un modèle reconnu en matière de sûreté et de sécurité. Alors que la guerre en Ukraine a provoqué la flambée des prix du mégawattheure et mis l’Europe sous tension, EDF n’a jamais aussi peu produit qu’en 2022, du fait des arr